# Pastizales Patagonia — Río Belgrano V2
Piloto reproducible. Enero 2026 frente a enero de 2018–2025 (8 años completos de Sentinel-2 SR). Cambiar MES para otros meses completos. No estima biomasa, especies ni capacidad de carga. El ráster original solo delimita el área válida y conserva la clasificación V1. Los umbrales V2 son exploratorios, no criterios agronómicos validados.

Fuentes: [INTA Santa Cruz](https://www.argentina.gob.ar/node/495331), [MARAS](https://www.argentina.gob.ar/noticias/maras-un-portal-para-seguir-de-cerca-el-estado-de-los-pastizales-en-patagonia), [Sentinel-2 SR](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED), [Cloud Score+](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_CLOUD_SCORE_PLUS_V1_S2_HARMONIZED).

In [ ]:
%pip -q install earthengine-api rasterio pyproj requests

In [ ]:
import ee, requests, rasterio, numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import files
from rasterio.warp import calculate_default_transform, reproject, Resampling, transform_bounds
from rasterio.transform import array_bounds
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path
import calendar, datetime, warnings, zipfile, shutil

PROYECTO = input('ID de tu proyecto de Google Cloud habilitado para Earth Engine: ').strip()
ee.Authenticate()
ee.Initialize(project=PROYECTO)
MES = 1
ANIO = 2026
BASE = list(range(2018, 2026))
MIN_OBS = 3
MIN_ANIOS = 6
assert ANIO not in BASE
assert datetime.date(ANIO,MES,calendar.monthrange(ANIO,MES)[1]) < datetime.date.today(), 'Elegir un mes completo'
print('Subí CLASES_NDVI_CAMPO.tif')
subidos = files.upload()
ruta = next(k for k in subidos if k.lower().endswith('.tif'))
salida = Path('resultados_v2'); salida.mkdir(exist_ok=True)
CRS = 'EPSG:32719'
with rasterio.open(ruta) as src:
    original = src.read(1, masked=True)
    valid = (~np.ma.getmaskarray(original) & np.isin(original.data,[1,2,3,4]))
    tr,w,h = calculate_default_transform(src.crs, CRS, src.width,src.height,*src.bounds,resolution=10)
    v1 = np.zeros((h,w),np.uint8)
    reproject(np.where(valid,original.data,0).astype('uint8'),v1,src_transform=src.transform,src_crs=src.crs,dst_transform=tr,dst_crs=CRS,resampling=Resampling.nearest,src_nodata=0,dst_nodata=0)
mask = v1>0
print('Superficie aproximada de la máscara a 10 m:',mask.sum()*0.01,'ha')
region = ee.Geometry.Rectangle(list(array_bounds(h,w,tr)),proj=CRS,geodesic=False)


In [ ]:
# Descargar compuestos mensuales reales, con filtros de nubes, sombras, nieve y agua clasificada.
# La exclusión de agua puede omitir sectores inundados; se informa cobertura válida.
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(region)
cs = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
def limpiar(img):
    scl = img.select('SCL')
    good = scl.eq(4).Or(scl.eq(5)).Or(scl.eq(7))
    good = good.And(img.select('cs_cdf').gte(0.60))
    return img.normalizedDifference(['B8','B4']).rename('ndvi').updateMask(good)
# Sentinel-2 usa B8 y B4 como nombres oficiales en Earth Engine.
def descargar(y):
    inicio=ee.Date.fromYMD(y,MES,1); fin=inicio.advance(1,'month')
    colec=s2.filterDate(inicio,fin).linkCollection(cs,['cs_cdf']).map(limpiar)
    if colec.size().getInfo()==0:
        return np.full((h,w),np.nan,np.float32)
    im=colec.median().updateMask(colec.count().gte(MIN_OBS)).unmask(-9999).toFloat()
    # Grilla fija UTM de 10 m; archivo pequeño, sin exportación asincrónica.
    url=im.getDownloadURL({'crs':CRS,'crs_transform':[tr.a,tr.b,tr.c,tr.d,tr.e,tr.f], 'dimensions':[w,h], 'format':'GEO_TIFF'})
    r=requests.get(url,timeout=240); r.raise_for_status()
    from rasterio.io import MemoryFile
    with MemoryFile(r.content) as mem, mem.open() as ds:
        if ds.shape != (h,w) or not ds.transform.almost_equals(tr) or ds.crs != rasterio.crs.CRS.from_string(CRS):
            raise RuntimeError('La grilla descargada no coincide')
        a=ds.read(1).astype('float32')
    a[(a<-1)|(a>1)|~mask]=np.nan
    return a
series={}
for y in BASE+[ANIO]:
    series[y]=descargar(y)
    print(y, 'cobertura válida:',round(100*np.isfinite(series[y]).sum()/mask.sum(),1),'%')


In [ ]:
# Referencia histórica: cada año pesa igual, comparación píxel a píxel.
hist=np.stack([series[y] for y in BASE]); actual=series[ANIO]
nvalid=np.isfinite(hist).sum(axis=0)
with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning)
    med=np.nanmedian(hist,axis=0)
    p25,p75=np.nanpercentile(hist,[25,75],axis=0)
valid_hist=mask & (nvalid>=MIN_ANIOS)
med[~valid_hist]=np.nan
ok=valid_hist & np.isfinite(actual)
if not ok.any(): raise RuntimeError('No hay cobertura histórica suficiente: revisar mes o filtros')
delta=np.where(ok,actual-med,np.nan)
# Umbrales históricos locales por píxel; igualdad se mantiene en rango habitual.
cl=np.zeros((h,w),np.uint8)
cl[ok]=2
cl[ok & (actual<p25)]=1
cl[ok & (actual>p75)]=3
# Actividad espacial histórica: cuartiles del propio campo, no calidad forrajera.
q=np.quantile(med[valid_hist],[.25,.50,.75]); print('Cuartiles locales NDVI:',q)
local=np.zeros((h,w),np.uint8); local[valid_hist]=1+np.searchsorted(q,med[valid_hist],side='left')
# Los empates pueden producir clases desiguales; no se fuerza su tamaño.
def tabla(arr,nombres):
    return pd.DataFrame([{'clase':k,'descripcion':v,'hectareas':float((arr==k).sum()*.01),'porcentaje_campo':float(100*(arr==k).sum()/mask.sum())} for k,v in nombres.items()])
t1=tabla(cl,{1:'Inferior al rango histórico',2:'Habitual',3:'Superior al rango histórico'})
t2=tabla(local,{1:'Cuartil inferior',2:'Cuartil medio bajo',3:'Cuartil medio alto',4:'Cuartil superior'})
print('Cobertura comparable:',round(100*ok.sum()/mask.sum(),2),'%')
display(t1,t2)
t1.to_csv(salida/'anomalias_hectareas.csv',index=False); t2.to_csv(salida/'actividad_hectareas.csv',index=False)
# Comparación V1 vs actividad histórica V2: no son equivalentes ni una validación.
pd.crosstab(v1[valid_hist],local[valid_hist]).mul(.01).to_csv(salida/'comparacion_v1_v2_ha.csv')


In [ ]:
def guardar(nombre,a,cat=False):
    nd=0 if cat else -9999
    a=np.where(np.isfinite(a),a,nd).astype('uint8' if cat else 'float32')
    with rasterio.open(salida/nombre,'w',driver='GTiff',height=h,width=w,count=1,dtype=a.dtype,crs=CRS,transform=tr,nodata=nd,compress='deflate') as ds: ds.write(a,1)
for nombre,a,cat in [('v1_original_10m.tif',v1,True),('actividad_local_v2.tif',local,True),('anomalia_clases.tif',cl,True),('anomalia_ndvi.tif',delta,False),('ndvi_historico.tif',med,False),('ndvi_actual.tif',actual,False)]: guardar(nombre,a,cat)
# Serie histórica sobre una máscara común para evitar sesgo por cobertura cambiante.
comun=mask & np.all(np.isfinite(np.stack(list(series.values()))),axis=0)
if comun.any():
    df=pd.DataFrame({'anio':list(series),'ndvi_medio':[float(a[comun].mean()) for a in series.values()]})
    df['cobertura_comun_pct']=100*comun.sum()/mask.sum(); df.to_csv(salida/'serie_historica.csv',index=False)
    df.plot(x='anio',y='ndvi_medio',marker='o',legend=False,title='NDVI medio — superficie común'); plt.ylabel('NDVI'); plt.tight_layout(); plt.savefig(salida/'serie.png',dpi=180); plt.show()
else: print('Sin superficie común entre todos los años; no se grafica una serie sesgada.')
for a,t,n,colors in [(v1,'Clasificación V1','mapa_v1',['#b87948','#e4cb70','#91b75c','#25663c']),(local,'Actividad vegetal relativa histórica V2','mapa_v2',['#b87948','#e4cb70','#91b75c','#25663c']),(cl,'Anomalía respecto de la referencia local','mapa_anomalia',['#b85035','#e0d7bc','#397d56'])]:
    fig,ax=plt.subplots(figsize=(10,8)); k=len(colors); im=ax.imshow(np.ma.masked_equal(a,0),cmap=ListedColormap(colors),norm=BoundaryNorm(np.arange(.5,k+1.5),k)); ax.set_title(t); ax.axis('off'); fig.colorbar(im,ax=ax,ticks=range(1,k+1),shrink=.65); fig.savefig(salida/(n+'.png'),dpi=200,bbox_inches='tight'); plt.show()
(salida/'metodologia.txt').write_text(f'Mes {MES}; objetivo {ANIO}; base {BASE}; mínimo observaciones {MIN_OBS}; mínimo años {MIN_ANIOS}; máscara derivada del ráster V1; grilla UTM 19S 10 m. Cuartiles locales y anomalías exploratorias; no equivalen a biomasa ni capacidad de carga. Área no evaluada: {(mask & ~ok).sum()*.01:.2f} ha. Cuartiles espaciales: {q}.',encoding='utf-8')
shutil.make_archive('pastizales_rio_belgrano_v2','zip',salida)
files.download('pastizales_rio_belgrano_v2.zip')